# Globally-Fitting (with Replicas) Experimental Data as Supervised Learning

## (1): Import Libraries:

In [1]:
import datetime
import gc
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Replica surrogate utilities (fit_scalers lives here)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', '..'))
from surrogate_model.replica_surrogate.replica_surrogate_lib import fit_scalers, ensemble_predict, build_replica_model

## (2): Matplotlib rcparams Configuration:

In [2]:
plt.rcParams.update({
    "text.usetex": True, "font.family": "serif",
})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 2.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 2.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['savefig.dpi'] = 300

## (3): Loading Data:

### (3.1): Loading Main File:

In [3]:
test_dataframe = pd.read_csv('../data/burner_data.csv')

### (3.2): Loading in the supervised learning $(x, y)$ pairs:

In [4]:
x_data = test_dataframe[["t", "x_b", "q_squared", "phi"]]
y_data = test_dataframe[["unp_beam_unp_target_xsec", "unp_target_bsa"]]

### (3.3): Checking out the $x$ data:

In [5]:
x_data.head()

,t,x_b,q_squared,phi
0,-0.11,0.185,1.62,-2.214125
1,-0.11,0.185,1.62,-1.963495
2,-0.11,0.185,1.62,-1.708154
3,-0.11,0.185,1.62,-1.445482
4,-0.11,0.185,1.62,-1.182810


### (3.4): Checking out the $y$ data:

In [6]:
y_data.head()

,unp_beam_unp_target_xsec,unp_target_bsa
0,6.8022,-0.069272
1,4.7934,-0.202153
2,3.4116,-0.162798
3,2.5726,-0.228718
4,1.6976,-0.213949


### (3.5): Splitting along training/validation/testing:

In [7]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = 0.1, shuffle = True)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = 0.1, shuffle = True)

### (3.6): Fitting StandardScalers on training data only

In [8]:
# Fit scalers ONCE on training data only — never on validation or test sets.
# Both observables are standardized so that under unweighted MSE they
# contribute equally to the gradient (xsec is O(5), BSA is O(0.2), so
# without standardization the BSA head collapses to a near-trivial fit).
x_scaler, y_scaler = fit_scalers(x_training, y_training)

# Standardize all splits using training-set statistics only (no data leakage)
x_training_scaled   = x_scaler.transform(x_training)
x_validation_scaled = x_scaler.transform(x_validation)
x_testing_scaled    = x_scaler.transform(x_testing)
x_data_scaled       = x_scaler.transform(x_data)

y_training_scaled   = y_scaler.transform(y_training)
y_validation_scaled = y_scaler.transform(y_validation)
y_testing_scaled    = y_scaler.transform(y_testing)

print(f"[INFO]: x_scaler  mean_  = {x_scaler.mean_}")
print(f"[INFO]: x_scaler  scale_ = {x_scaler.scale_}")
print(f"[INFO]: y_scaler  mean_  = {y_scaler.mean_}")
print(f"[INFO]: y_scaler  scale_ = {y_scaler.scale_}")

[INFO]: x_scaler  mean_  = [-0.11        0.185       1.62       -0.16842427]
[INFO]: x_scaler  scale_ = [1.         1.         1.         1.32675845]
[INFO]: y_scaler  mean_  = [ 2.9095    -0.0082678]
[INFO]: y_scaler  scale_ = [2.02236268 0.17681366]


In [9]:
len(x_training)

14

In [10]:
len(x_validation)

2

In [11]:
len(x_testing)

2

## (4): Defining the DNN Model:

In [12]:
# Model factory moved to replica_surrogate_lib.build_replica_model

## (5): **The Main Part of the Program: Replica Method!**

In [ ]:
number_of_replicas = 100

all_histories = []

models = []

for index in range(number_of_replicas):
    replica_number = index + 1
    print(f"[INFO]: Now training replica #{replica_number}")

    replica_seed = replica_number
    tf.random.set_seed(replica_seed)
    np.random.seed(replica_seed)

    tf.keras.backend.clear_session()
    gc.collect()

    dnn_model = build_replica_model(replica_seed)

    # Train on standardized targets so both xsec and BSA contribute equally
    # to the unweighted MSE loss.
    dnn_model_history = dnn_model.fit(
        x_training_scaled, y_training_scaled,
        validation_data = (x_validation_scaled, y_validation_scaled),
        epochs = 1000, 
        # [NOTE]: BATCHSIZE really matters!
        # batch_size = len(x_training),
        batch_size = 8,
        callbacks = [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor = "val_loss", factor = 0.5, patience = 50, min_lr = 1e-6,
                verbose = 1),
            tf.keras.callbacks.EarlyStopping(
                monitor = "val_loss", patience = 100, restore_best_weights = True
            )
        ],
        verbose = 0)
    
    all_histories.append(dnn_model_history.history)

    model_testing_loss = dnn_model.evaluate(x_testing_scaled, y_testing_scaled, verbose = 0)
    print(f"[INFO]: Test loss for replica #{replica_number}: {model_testing_loss}")

    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(dnn_model_history.history['loss'], 
        label = "Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(dnn_model_history.history['val_loss'], 
        label = "Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Loss (standardized space)", fontsize = 14.)
    axis.set_title(
        rf"Surrogate Model (Testing = ${model_testing_loss:.3f}$)",
        fontsize = 14.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = f"./plots/surrogate_lc_replica_{replica_number}_v13.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(figure)

    del figure
    del axis

    # dnn_model.save(f"replica_{replica_number}_v13.keras")
    models.append(dnn_model)

# Ensemble predict on all data points in physical units
average_prediction, standard_dev_prediction = ensemble_predict(
    models, x_data, x_scaler, y_scaler
)

[INFO]: Now training replica #1


2026-05-26 22:47:33.569556: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz



Epoch 367: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.

Epoch 417: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
[INFO]: Test loss for replica #1: 0.13608410954475403


[INFO]: Now training replica #2

Epoch 380: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.

Epoch 430: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
[INFO]: Test loss for replica #2: 0.12501585483551025


[INFO]: Now training replica #3

Epoch 342: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.

Epoch 392: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
[INFO]: Test loss for replica #3: 0.10913459956645966


...

[INFO]: Now training replica #50



Epoch 369: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.

Epoch 419: ReduceLROnPlateau reducing learning rate to 7.500000356230885e-05.
[INFO]: Test loss for replica #50: 0.12193544954061508


1/1 [==============================] - 0s 20ms/step


1/1 [==============================] - 0s 21ms/step


1/1 [==============================] - 0s 20ms/step


In [14]:
grouped = test_dataframe.groupby(['t', 'x_b', 'q_squared'])

In [15]:
# this is trento convention: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361)
special_phis = [0, np.pi/2, -np.pi/2, np.pi]

for (t_value, xb_value, qsquared_value), group in grouped:
    print(f"Processing t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values('phi')

    xsec_err = group['unp_beam_unp_target_xsec_err'].values
    bsa_err = group['unp_target_bsa_err'].values

    indices = group.index.values

    # all_predictions already in physical units (inverse-transformed in the loop above)
    xsec_pred = average_prediction[indices, 0]
    bsa_pred = average_prediction[indices, 1]
    xsec_std = standard_dev_prediction[indices, 0]
    bsa_std = standard_dev_prediction[indices, 1]

    x_smooth = np.column_stack([
        np.full_like(phi_smooth, t_value),
        np.full_like(phi_smooth, xb_value),
        np.full_like(phi_smooth, qsquared_value),
        phi_smooth
    ])
    # Ensemble predict on smooth curve inputs (x_scaler applied inside)
    smooth_mean, smooth_std = ensemble_predict(models, x_smooth, x_scaler, y_scaler)

    xsec_smooth_mean = smooth_mean[:, 0]
    bsa_smooth_mean = smooth_mean[:, 1]

    xsec_smooth_std = smooth_std[:, 0]
    bsa_smooth_std = smooth_std[:, 1]

    for phi_target in special_phis:
        phi_index = np.argmin(np.abs(phi_smooth - phi_target))
        phi_actual = phi_smooth[phi_index]
        sigma_value = bsa_smooth_std[phi_index]
        print(
            f"[INFO]: "
            f"(xb = {xb_value:.3f}, "
            f"t = {t_value:.3f}, "
            f"Q2 = {qsquared_value:.3f}) "
            f"BSA 1σ uncertainty at "
            f"phi = {phi_actual:.3f} rad "
            f"is ±{sigma_value:.6f}"
        )

    phi = group['phi'].values
    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res = bsa_actual - bsa_pred

    chi2_xsec = np.sum(xsec_res**2) / len(phi)
    chi2_bsa = np.sum(bsa_res**2) / len(phi)

    residuals_figure, axes = plt.subplots(2, 2, figsize = (10, 8), sharex = 'col', layout = "tight")

    axes[1, 0].text(
        -0.1, -0.1, 
        fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axes[1, 0].transAxes)

    axes[0, 0].plot(phi_smooth, xsec_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[0, 0].fill_between(
        phi_smooth, xsec_smooth_mean - xsec_smooth_std, xsec_smooth_mean + xsec_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    axes[0, 0].errorbar(
        phi, xsec_actual, yerr = xsec_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    axes[0, 0].set_ylabel(r"$d^{4}\sigma$", fontsize = 14)
    axes[0, 0].set_title(rf"Cross Section ($\chi^2_\nu = {chi2_xsec:.3f}$)")
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle=':', alpha = 0.6)

    axes[0, 1].scatter(phi, xsec_res, color = 'blue', alpha = 0.6)
    axes[0, 1].axhline(0, color = 'black', linestyle = '--')
    axes[0, 1].set_title("Residuals")
    axes[0, 1].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 0].plot(phi_smooth, bsa_smooth_mean, color = 'green', lw = 2, label = rf'Replica Average ($N = {number_of_replicas}$)')
    axes[1, 0].fill_between(
        phi_smooth, bsa_smooth_mean - bsa_smooth_std, bsa_smooth_mean + bsa_smooth_std,
        color = 'green', alpha = 0.3,
        label = r'$\sigma$ band'
    )
    axes[1, 0].errorbar(
        phi, bsa_actual, yerr = bsa_err, 
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    
    axes[1, 0].set_ylabel("BSA", fontsize = 14)
    axes[1, 0].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 0].set_title(rf"BSA ($\chi^2_\nu = {chi2_bsa:.3f}$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle = ':', alpha = 0.6)

    axes[1, 1].scatter(phi, bsa_res, color = 'purple', alpha = 0.6)
    axes[1, 1].axhline(0, color = 'black', linestyle = '--')
    axes[1, 1].set_xlabel(r"$\phi$ (radians)", fontsize = 14)
    axes[1, 1].set_title("Residuals")
    axes[1, 1].grid(True, linestyle = ':', alpha = 0.6)
    
    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$t = {t_value:.3f}$, $x_\textrm{{B}} = {xb_value:.3f}$, $Q^2 = {qsquared_value:.3f}$",
        fontsize = 16
    )

    filename = f"./plots/t{t_value:.3f}_xb{xb_value:.3f}_q2{qsquared_value:.3f}_residuals_v13"
    for extension in ['png', 'eps']:
        residuals_figure.savefig(
            fname = f"{filename}.{extension}",
            facecolor = 'white', transparent = False)

    plt.close(residuals_figure)

Processing t = -0.11, xb = 0.185, Q2 = 1.62
 1/12 [=>............................] - ETA: 0s

/opt/anaconda3/envs/gpd_dnn/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


12/12 [==============================] - 0s 263us/step
[INFO]: (xb = 0.185, t = -0.110, Q2 = 1.620) BSA 1σ uncertainty at phi = 0.000 rad is ±0.019174
[INFO]: (xb = 0.185, t = -0.110, Q2 = 1.620) BSA 1σ uncertainty at phi = 1.571 rad is ±0.038021
[INFO]: (xb = 0.185, t = -0.110, Q2 = 1.620) BSA 1σ uncertainty at phi = -1.571 rad is ±0.019329
[INFO]: (xb = 0.185, t = -0.110, Q2 = 1.620) BSA 1σ uncertainty at phi = 3.142 rad is ±0.020970


In [16]:
xb_q2_groups = test_dataframe.groupby(['x_b', 'q_squared'])
n_xb_q2 = xb_q2_groups.ngroups
print(n_xb_q2)

1


In [17]:
phi_grid = np.linspace(-np.pi, np.pi, 361)

In [18]:
for (xb_value, qsquared_value), group in xb_q2_groups:

    group = group.sort_values(['t', 'phi'])

    t_values = np.sort(group['t'].unique())

    phi_meshgrid, t_meshgrid = np.meshgrid(phi_grid, t_values)

    phi_data = group['phi'].values
    t_data = group['t'].values

    indices = group.index.values

    # all_predictions already in physical units
    xsec_pred = average_prediction[indices, 0]
    bsa_pred = average_prediction[indices, 1]

    xsec_actual = group['unp_beam_unp_target_xsec'].values
    bsa_actual = group['unp_target_bsa'].values

    xsec_res = xsec_actual - xsec_pred
    bsa_res = bsa_actual - bsa_pred

    colors_xsec = np.where(xsec_res >= 0, 'red', 'blue')
    colors_bsa = np.where(bsa_res >= 0, 'red', 'blue')

    model_surface_input = np.column_stack([
        t_meshgrid.ravel(),
        np.full(t_meshgrid.size, xb_value),
        np.full(t_meshgrid.size, qsquared_value),
        phi_meshgrid.ravel()
    ])
    # Ensemble predict on surface inputs (x_scaler applied inside)
    surface_mean, surface_std_dev = ensemble_predict(
        models, model_surface_input, x_scaler, y_scaler
    )

    xsec_surface = surface_mean[:, 0].reshape(t_meshgrid.shape)
    bsa_surface = surface_mean[:, 1].reshape(t_meshgrid.shape)

    xsec_stddev_surface = surface_std_dev[:, 0].reshape(t_meshgrid.shape)
    bsa_stddev_surface = surface_std_dev[:, 1].reshape(t_meshgrid.shape)

    zero_plane_xsec = np.zeros_like(xsec_surface)
    zero_plane_bsa = np.zeros_like(bsa_surface)

    fig = plt.figure(figsize = (10, 10), layout = "tight")

    ax1 = fig.add_subplot(2, 2, 1, projection = '3d')
    ax2 = fig.add_subplot(2, 2, 2, projection = '3d')
    ax3 = fig.add_subplot(2, 2, 3, projection = '3d')
    ax4 = fig.add_subplot(2, 2, 4, projection = '3d')

    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface, cmap = 'viridis', alpha = 0.5)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface + xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.plot_surface(phi_meshgrid, t_meshgrid, xsec_surface - xsec_stddev_surface, color = "red", alpha = 0.2)
    ax1.scatter(phi_data, t_data, xsec_actual, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax1.set_xlabel(r'$\phi$ (Radians)')
    ax1.set_ylabel(r'$t$')
    ax1.set_zlabel(r'$d^{4}\sigma^{UU}$')
    ax1.set_title('Cross Section', fontsize = 16)

    ax2.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_xsec, color = 'gray', alpha = 0.15)
    ax2.scatter(phi_data, t_data, xsec_res, color = colors_xsec, s = 20)
 
    ax2.set_xlabel(r'$\phi$')
    ax2.set_ylabel(r'$t$')
    ax2.set_zlabel('Residuals')
    ax2.set_title('Cross Section Residuals', fontsize = 16)

    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface, cmap='plasma', alpha = 0.5)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface + bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.plot_surface(phi_meshgrid, t_meshgrid, bsa_surface - bsa_stddev_surface, color = "red", alpha = 0.2)
    ax3.scatter(phi_data, t_data, bsa_actual, facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

    ax3.set_xlabel(r'$\phi$ (Radians)')
    ax3.set_ylabel(r'$t$')
    ax3.set_zlabel('BSA')
    ax3.set_title('BSA', fontsize = 16)

    ax4.plot_surface(phi_meshgrid, t_meshgrid, zero_plane_bsa, color = 'gray', alpha = 0.15)
    ax4.scatter(phi_data, t_data, bsa_res, color = colors_bsa, s = 20)

    ax4.set_xlabel(r'$\phi$')
    ax4.set_ylabel(r'$t$')
    ax4.set_zlabel('Residuals')
    ax4.set_title('BSA Residuals', fontsize = 16)

    fig.suptitle(
        r"DNN Interpolations Across $t$ and $\phi$"
        "\n"
        rf"Kinematic Setting: $x_\textrm{{B}} = {xb_value:.4g}$, $Q^2 = {qsquared_value:.4g}$",
        fontsize = 16)

    plot_filename = f"./plots/surface_xb{xb_value:.4g}_q2{qsquared_value:.4g}_v13"
    
    for extension in ['png', 'eps']:
        fig.savefig(f"{plot_filename}.{extension}", facecolor = 'white')

    plt.close(fig)

    # cleanup:
    del fig
    del ax1
    del ax2
    del ax3
    del ax4

12/12 [==============================] - 0s 264us/step


/opt/anaconda3/envs/gpd_dnn/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


12/12 [==============================] - 0s 271us/step
